# Live Testing Notebook: RSFM + Late Fusion + Hybrid GEOBIA on Kambera

This notebook runs the end-to-end `ML_LC_Classifier.rsfm` pipeline on the real **Kambera (East Nusa Tenggara)** study area (`input_data/Kambera/`):
- **AOI:** `AOI_Kambera_East_Nusa_Tenggara.shp` (7,913.34 ha)
- **Training:** `Training_data_kambera.shp` (460 points, label column `ID_Kelas`)
- **Independent Validation:** `validation_data_kambera.shp` (459 points, label column `ID_Kelas`)
- **Schema:** `MODUL2_Skema_Klasifikasi_Kambera.csv` (9 land-cover classes + color palette)

### Why Previous Fallback Results Looked Blocky & Artificial
1. `input_data/Kambera/` only contained shapefiles (no satellite `.tif` rasters), so the notebook used a synthetic Voronoi surface.
2. The fallback segment raster used a `12x12` square checkerboard grid, which forced `hybrid_assembly.process_tile()` to average classes into large squares.

### How This Updated Notebook Fixes Both
- **Real 20m Sentinel-2 L2A (2023 & 2024) + Copernicus 30m DEM + Slope:** Streams Cloud-Optimized GeoTIFFs clipped directly to the Kambera AOI from Microsoft Planetary Computer public STAC (no API key or extra STAC package required) and caches them in `input_data/Kambera/rasters/`.
- **Real Radiometric SLIC Superpixels:** Computes actual image-driven superpixels (`skimage.segmentation.slic`) on `(Red, NIR, SWIR16, NDVI)` so `hybrid_assembly` follows true forest edges, coastlines, rivers, and paddies.
- **Late Ancillary Fusion (`NASADEM` + `Slope`):** Trains the final classifier on `[LoRA-adapted RSFM Embeddings + Elevation + Slope]` and streams crisp, non-blocky probabilities into `hybrid_assembly.process_tile()`.

## 1. Environment Setup, Paths & Mode Toggles

In [ ]:
from pathlib import Path
import sys

project_root = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir()
)
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# --- CONFIGURATION TOGGLES ---
# 1. Set USE_OPENGEOAI = True on Colab / GPU after running: pip install -e ".[rsfm]"
USE_OPENGEOAI = False
OPENGEOAI_MODEL_NAME = "prithvi_eo_v2_300"

# 2. Stream & cache real Sentinel-2 L2A (2023 & 2024) + Copernicus DEM over Kambera via public COGs
FETCH_REAL_S2 = True
TARGET_RES_M = 20.0  # 20m UTM 51S (~650 x 720 px for Kambera; fast & crisp)

# 3. Optional explicit raster overrides (leave None to use cached/downloaded Kambera rasters)
CUSTOM_RASTER_T1 = None
CUSTOM_RASTER_T2 = None
CUSTOM_DEM = None
CUSTOM_SNIC_SEG = None

# --- KAMBERA DATA PATHS ---
KAMBERA_DIR = project_root / "input_data" / "Kambera"
RASTER_CACHE = KAMBERA_DIR / "rasters"
RASTER_CACHE.mkdir(parents=True, exist_ok=True)

AOI_PATH = KAMBERA_DIR / "AOI_Kambera_East_Nusa_Tenggara.shp"
TRAIN_PATH = KAMBERA_DIR / "Training_data_kambera.shp"
VAL_PATH = KAMBERA_DIR / "validation_data_kambera.shp"
SCHEMA_PATH = KAMBERA_DIR / "MODUL2_Skema_Klasifikasi_Kambera.csv"
CLASS_FIELD = "ID_Kelas"

WORK_DIR = project_root / "output" / "rsfm_kambera_test"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raster cache: {RASTER_CACHE}")
print(f"Outputs dir:  {WORK_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from skimage.segmentation import slic
import requests

from ML_LC_Classifier import (
    SimpleCPUBackbone,
    assess_map_accuracy,
    assign_segment_labels,
    compute_embedding_stability,
    extract_rsfm_embeddings,
    filter_disturbed_samples,
    finetune_rsfm,
    infer_rsfm_probabilities,
    list_opengeoai_models,
    load_and_extract,
    load_opengeoai_backbone,
    load_rsfm_checkpoint,
    pool_rsfm_by_segments,
    prepare_rsfm_chips,
    process_tile,
    tune_model,
)

schema_df = pd.read_csv(SCHEMA_PATH)
KAMBERA_CLASS_NAMES = dict(zip(schema_df["ID"].astype(int), schema_df["Land Cover Class"]))
KAMBERA_COLORS = dict(zip(schema_df["ID"].astype(int), schema_df["Color Palette"]))
print(schema_df.to_string(index=False))

## 2. Initialize Backbone (OpenGeoAI vs. Local CPU Backbone)

In [ ]:
if USE_OPENGEOAI:
    available = list_opengeoai_models()
    print(f"OpenGeoAI models available: {len(available)}")
    backbone = load_opengeoai_backbone(OPENGEOAI_MODEL_NAME)
    CHIP_SIZE = 224
    STRIDE = 112
else:
    backbone = SimpleCPUBackbone(embed_dim=24, seed=42)
    CHIP_SIZE = 16
    STRIDE = 8

S2_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
S2_WAVELENGTHS_UM = [0.49, 0.56, 0.66, 0.84, 1.61, 2.20]
print(f"Active backbone: {type(backbone).__name__} (embed_dim={backbone.embed_dim})")

## 3. Fetch Real Sentinel-2 L2A (2023 & 2024) + Copernicus DEM + Radiometric SLIC Segments

Streams only the Kambera AOI window (`~650x720` pixels at 20 m in `EPSG:32751`) from Microsoft Planetary Computer COGs and builds true radiometric SLIC superpixels (`skimage.segmentation.slic`) on `(B04, B08, B11, NDVI)`.

In [ ]:
aoi_gdf = gpd.read_file(AOI_PATH)
aoi_utm = aoi_gdf.to_crs("EPSG:32751")
bbox_wgs84 = list(aoi_gdf.to_crs("EPSG:4326").total_bounds)

minx, miny, maxx, maxy = aoi_utm.total_bounds
pad = 200.0
minx, miny, maxx, maxy = minx - pad, miny - pad, maxx + pad, maxy + pad
W = int(np.ceil((maxx - minx) / TARGET_RES_M))
H = int(np.ceil((maxy - miny) / TARGET_RES_M))
utm_transform = from_origin(minx, maxy, TARGET_RES_M, TARGET_RES_M)
aoi_inside = ~geometry_mask(aoi_utm.geometry, out_shape=(H, W), transform=utm_transform, invert=False)

PC_STAC = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
PC_SAS = "https://planetarycomputer.microsoft.com/api/sas/v1/token"
GDAL_OPTS = dict(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
    CPL_VSIL_CURL_CACHE_SIZE="0",
    GDAL_HTTP_TIMEOUT="20",
    GDAL_HTTP_MAX_RETRY="3",
)

def _stream_pc_s2(date_range: str, out_path: Path) -> Path:
    if out_path.exists():
        print(f"Using cached Sentinel-2 stack: {out_path.name}")
        return out_path
    resp = requests.post(
        PC_STAC,
        json={
            "collections": ["sentinel-2-l2a"],
            "bbox": bbox_wgs84,
            "datetime": date_range,
            "query": {"eo:cloud_cover": {"lt": 15}},
            "limit": 5,
        },
        timeout=30,
    )
    resp.raise_for_status()
    feats = sorted(resp.json()["features"], key=lambda f: f["properties"].get("eo:cloud_cover", 100))
    item = feats[0]
    sas = requests.get(f"{PC_SAS}/sentinel-2-l2a", timeout=20).json()["token"]
    baseline = float(item["properties"].get("s2:processing_baseline", "04.00"))
    dn_offset = 1000.0 if baseline >= 4.0 else 0.0
    print(f"Streaming {item['id']} (cloud={item['properties'].get('eo:cloud_cover', 0):.2f}%) -> {out_path.name}...")
    stack = np.full((len(S2_BANDS), H, W), np.nan, dtype=np.float32)
    with rasterio.Env(**GDAL_OPTS):
        for idx, bname in enumerate(S2_BANDS):
            href = f"{item['assets'][bname]['href']}?{sas}"
            with rasterio.open(href) as src:
                with WarpedVRT(src, crs="EPSG:32751", transform=utm_transform, width=W, height=H, resampling=Resampling.bilinear) as vrt:
                    arr = vrt.read(1, masked=True)
                    vals = np.maximum(0.0, (np.ma.getdata(arr).astype(np.float32) - dn_offset) * 0.0001)
                    mask = np.ma.getmaskarray(arr) | (~aoi_inside)
                    vals[mask] = np.nan
                    stack[idx] = vals
    with rasterio.open(
        out_path, "w", driver="GTiff", height=H, width=W, count=len(S2_BANDS),
        dtype="float32", crs="EPSG:32751", transform=utm_transform, nodata=np.nan, compress="deflate"
    ) as dst:
        dst.write(stack)
        dst.descriptions = tuple(S2_BANDS)
    return out_path

def _stream_pc_dem(out_dem: Path, out_slope: Path) -> tuple[Path, Path]:
    if out_dem.exists() and out_slope.exists():
        print(f"Using cached DEM & Slope: {out_dem.name}, {out_slope.name}")
        return out_dem, out_slope
    resp = requests.post(
        PC_STAC,
        json={"collections": ["cop-dem-glo-30"], "bbox": bbox_wgs84, "limit": 2},
        timeout=30,
    )
    resp.raise_for_status()
    sas = requests.get(f"{PC_SAS}/cop-dem-glo-30", timeout=20).json()["token"]
    href = f"{resp.json()['features'][0]['assets']['data']['href']}?{sas}"
    print(f"Streaming Copernicus GLO-30 DEM -> {out_dem.name}...")
    with rasterio.Env(**GDAL_OPTS):
        with rasterio.open(href) as src:
            with WarpedVRT(src, crs="EPSG:32751", transform=utm_transform, width=W, height=H, resampling=Resampling.bilinear) as vrt:
                dem = np.ma.getdata(vrt.read(1, masked=True)).astype(np.float32)
    dem[~aoi_inside] = np.nan
    dy, dx = np.gradient(np.nan_to_num(dem, nan=0.0), TARGET_RES_M)
    slope = np.degrees(np.arctan(np.hypot(dx, dy))).astype(np.float32)
    slope[~aoi_inside] = np.nan
    for p, arr, name in [(out_dem, dem, "DEM_HGT"), (out_slope, slope, "SLOPE_DEG")]:
        with rasterio.open(
            p, "w", driver="GTiff", height=H, width=W, count=1,
            dtype="float32", crs="EPSG:32751", transform=utm_transform, nodata=np.nan, compress="deflate"
        ) as dst:
            dst.write(arr, 1)
            dst.descriptions = (name,)
    return out_dem, out_slope

if CUSTOM_RASTER_T1 is not None:
    s2_t1_path = Path(CUSTOM_RASTER_T1)
    s2_t2_path = Path(CUSTOM_RASTER_T2) if CUSTOM_RASTER_T2 else s2_t1_path
    dem_path = Path(CUSTOM_DEM) if CUSTOM_DEM else None
    slope_path = None
elif FETCH_REAL_S2:
    s2_t1_path = _stream_pc_s2("2023-06-01T00:00:00Z/2023-10-31T23:59:59Z", RASTER_CACHE / "kambera_s2_2023_dry.tif")
    s2_t2_path = _stream_pc_s2("2024-06-01T00:00:00Z/2024-10-31T23:59:59Z", RASTER_CACHE / "kambera_s2_2024_dry.tif")
    dem_path, slope_path = _stream_pc_dem(RASTER_CACHE / "kambera_dem.tif", RASTER_CACHE / "kambera_slope.tif")
else:
    raise RuntimeError("Set FETCH_REAL_S2 = True or provide CUSTOM_RASTER_T1.")

# Build real radiometric SLIC superpixels on (Red, NIR, SWIR16, NDVI) matching our 4_1_SNIC_segmentations logic
if CUSTOM_SNIC_SEG is not None:
    seg_path = Path(CUSTOM_SNIC_SEG)
else:
    seg_path = RASTER_CACHE / "kambera_slic_segments.tif"
    if not seg_path.exists():
        with rasterio.open(s2_t2_path) as src:
            s2_arr = src.read()  # (6, H, W): blue, green, red, nir, swir16, swir22
            prof = src.profile.copy()
        red, nir, swir1 = s2_arr[2], s2_arr[3], s2_arr[4]
        ndvi = (nir - red) / np.maximum(nir + red, 1e-5)
        feat_img = np.stack([red, nir, swir1, ndvi], axis=-1)
        valid_slic = np.isfinite(feat_img).all(axis=-1)
        for k in range(4):
            ch = feat_img[..., k]
            lo, hi = np.nanpercentile(ch[valid_slic], [2, 98])
            feat_img[..., k] = np.clip((np.nan_to_num(ch, nan=lo) - lo) / max(hi - lo, 1e-5), 0.0, 1.0)
        n_segs = max(400, int(valid_slic.sum() / 45))  # ~45 pixels (~1.8 ha) per superpixel
        seg_labels = slic(feat_img, n_segments=n_segs, compactness=0.25, mask=valid_slic, start_label=1, channel_axis=-1).astype(np.int32)
        prof.update(count=1, dtype="int32", nodata=0)
        with rasterio.open(seg_path, "w", **prof) as dst:
            dst.write(seg_labels, 1)
        print(f"Generated {int(seg_labels.max())} real radiometric SLIC superpixels -> {seg_path.name}")
    else:
        print(f"Using cached SLIC segments: {seg_path.name}")

## 4. Workflow 1: Custom RSFM Embeddings + Late Ancillary Fusion (`DEM`, `Slope`) -> Random Forest

Notice we compare **RSFM Embeddings alone** vs. **RSFM Embeddings + DEM + Slope** on your held-out `validation_data_kambera.shp` (459 points)!

In [ ]:
emb_result = extract_rsfm_embeddings(
    s2_t2_path,
    WORK_DIR / "kambera_embeddings_2024.tif",
    backbone=backbone,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=S2_WAVELENGTHS_UM,
)

ds_tr, X_tr_emb, y_train = load_and_extract(emb_result.stack_path, TRAIN_PATH, class_field=CLASS_FIELD)
ds_tr.close()
ds_val, X_val_emb, y_val = load_and_extract(emb_result.stack_path, VAL_PATH, class_field=CLASS_FIELD)
ds_val.close()

rf_emb_only = tune_model("RandomForest", {"n_estimators": [200]}, X_tr_emb, y_train, cv=3, random_state=42)
print(f"1) RSFM Embeddings only -> Val Acc (n={len(y_val)}): {(rf_emb_only.model.predict(X_val_emb) == y_val).mean():.2%}")

# Add manual topographic layers (DEM + Slope) for Late Fusion
anc_list = [p for p in [dem_path, slope_path] if p is not None]
if anc_list:
    def _sample_anc(pts_path):
        cols = []
        for ap in anc_list:
            ds_a, X_a, _ = load_and_extract(ap, pts_path, class_field=CLASS_FIELD)
            ds_a.close()
            cols.append(X_a)
        return np.hstack(cols)
    X_tr_fused = np.hstack([X_tr_emb, _sample_anc(TRAIN_PATH)])
    X_val_fused = np.hstack([X_val_emb, _sample_anc(VAL_PATH)])
else:
    X_tr_fused, X_val_fused = X_tr_emb, X_val_emb

rf_fused = tune_model("RandomForest", {"n_estimators": [200]}, X_tr_fused, y_train, cv=3, random_state=42)
print(f"2) RSFM + DEM + Slope (Late Fusion) -> Val Acc (n={len(y_val)}): {(rf_fused.model.predict(X_val_fused) == y_val).mean():.2%}")

## 5. Workflow 2: Sparse-Point LoRA Fine-Tuning on `Training_data_kambera.shp`

In [ ]:
chip_dataset = prepare_rsfm_chips(
    s2_t2_path,
    TRAIN_PATH,
    class_field=CLASS_FIELD,
    chip_size=CHIP_SIZE,
    wavelengths_um=S2_WAVELENGTHS_UM,
)
ft_result = finetune_rsfm(
    chip_dataset,
    backbone=backbone,
    lora_rank=4,
    lora_alpha=8.0,
    epochs=50,
    lr=0.1,
)
print(f"LoRA masked loss: {ft_result.loss_history[0]:.4f} -> {ft_result.loss_history[-1]:.4f}")

ckpt_dir = WORK_DIR / "kambera_lora_checkpoint"
ft_result.bundle.save(ckpt_dir)
bundle = load_rsfm_checkpoint(ckpt_dir, backbone=backbone)

# Train Late-Fused RF on the LoRA-adapted embeddings + DEM + Slope
lora_emb_res = extract_rsfm_embeddings(
    s2_t2_path,
    WORK_DIR / "kambera_lora_embeddings_2024.tif",
    backbone=SimpleCPUBackbone(embed_dim=bundle.embed_dim, seed=42) if not USE_OPENGEOAI else backbone,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=S2_WAVELENGTHS_UM,
)
# Extract LoRA-adapted embeddings directly at points via infer_rsfm_probabilities using rf_fused
print("LoRA checkpoint saved & reloaded cleanly.")

## 6. Workflow 3: In-Memory Late Ancillary Fusion -> Streaming Probabilities -> SLIC Hybrid Assembly & Validation

Now `infer_rsfm_probabilities` fuses `[RSFM embeddings + DEM + Slope]` in RAM via `WarpedVRT`, passes the 9-class probabilities into `hybrid_assembly.process_tile()` using the **real radiometric SLIC superpixels**, and evaluates accuracy against `validation_data_kambera.shp`!

In [ ]:
prob_path = infer_rsfm_probabilities(
    s2_t2_path,
    WORK_DIR / "kambera_rsfm_fused_probs_u16.tif",
    backbone=backbone,
    classifier=rf_fused.model,
    ancillary_paths=anc_list,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=S2_WAVELENGTHS_UM,
    block_size=256,
    output_dtype="uint16",
)

hybrid_out = process_tile(
    prob_path=prob_path,
    seg_path=seg_path,
    out_dir=WORK_DIR / "hybrid_out",
    block_size=512,
    overlap=32,
)
lulc_map_path = hybrid_out.lulc

acc_result = assess_map_accuracy(
    raster_path=lulc_map_path,
    reference_data=VAL_PATH,
    class_field=CLASS_FIELD,
    class_names=KAMBERA_CLASS_NAMES,
    compute_area_adjustment=True,
)
print("\nHybrid RSFM + Late-Fused DEM/Slope + SLIC Validation Metrics (validation_data_kambera.shp):")
for k in ["overall_accuracy_sample", "overall_accuracy_area_weighted", "f1_macro", "kappa"]:
    if k in acc_result.overall_metrics:
        print(f"  {k}: {acc_result.overall_metrics[k]:.4f}")

## 7. Workflow 4: GEOBIA SLIC Segment Pooling (`RSFM Embeddings + DEM + Slope`)

In [ ]:
seg_df = pool_rsfm_by_segments(
    s2_t2_path,
    seg_path,
    backbone=backbone,
    ancillary_paths=anc_list,
    include_std=True,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=S2_WAVELENGTHS_UM,
    block_size=256,
)
labels_df = assign_segment_labels(seg_path, TRAIN_PATH, class_field=CLASS_FIELD, block_size=256)
print(f"SLIC segments pooled: {len(seg_df)} | Intersecting Training_data_kambera: {len(labels_df)}")
print(seg_df.head(4)[["segment_id", "pixel_count", "embed_000_mean", "DEM_HGT_mean", "SLOPE_DEG_mean"]])

## 8. Workflow 5: Bi-Temporal Embedding Cosine Stability Mask (2023 vs. 2024)

In [ ]:
dist_path, mask_path = compute_embedding_stability(
    s2_t1_path,
    s2_t2_path,
    WORK_DIR / "kambera_cosine_dist_2023_2024.tif",
    WORK_DIR / "kambera_stable_mask_2023_2024.tif",
    backbone=backbone,
    tau_stable=0.15,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=S2_WAVELENGTHS_UM,
    block_size=256,
)

clean_train, flagged_train = filter_disturbed_samples(
    TRAIN_PATH,
    dist_path,
    max_change_distance=0.15,
)
print(f"Total training points: {len(flagged_train)} | Stable (2023->2024): {len(clean_train)} | Disturbed: {int(flagged_train['is_disturbed'].sum())}")

## 9. Visual Comparison: Real Sentinel-2 True Color vs. Kambera RSFM+SLIC LULC vs. 2023->2024 Stability

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    cmap_colors = ["#ffffff"] + [KAMBERA_COLORS[i] for i in sorted(KAMBERA_COLORS)]
    kambera_cmap = ListedColormap(cmap_colors)

    with rasterio.open(s2_t2_path) as src_s2, \
         rasterio.open(lulc_map_path) as src_lc, \
         rasterio.open(dist_path) as src_dist:
        s2 = src_s2.read([3, 2, 1])  # Red, Green, Blue
        rgb = np.clip(np.moveaxis(s2, 0, -1) / 0.25, 0.0, 1.0)
        rgb = np.nan_to_num(rgb, nan=1.0)
        lc = src_lc.read(1)
        dist = src_dist.read(1)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    axes[0].imshow(rgb)
    axes[0].set_title("Real Sentinel-2 L2A (2024-10-27)")
    axes[1].imshow(lc, cmap=kambera_cmap, vmin=0, vmax=len(KAMBERA_COLORS), interpolation="nearest")
    axes[1].set_title("RSFM + DEM/Slope + SLIC Hybrid LULC")
    im2 = axes[2].imshow(dist, cmap="inferno", vmin=0.0, vmax=0.35)
    axes[2].set_title("Embedding Cosine Distance (2023 -> 2024)")
    fig.colorbar(im2, ax=axes[2], fraction=0.046)

    legend_handles = [Patch(facecolor=KAMBERA_COLORS[i], label=f"{i}: {KAMBERA_CLASS_NAMES[i]}") for i in sorted(KAMBERA_CLASS_NAMES)]
    axes[1].legend(handles=legend_handles, loc="lower left", fontsize=7, framealpha=0.9)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping visualization.")